<a href="https://colab.research.google.com/github/acoiman/pdt/blob/main/asthma_mortality/notebooks/Python/14.Asthma_Mortality_RF_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📈 🚧 Random Forest Regression-Hurdle-type Modelling



Hurdle-type modeling is a statistical approach designed to analyze count data, particularly when there is an excess of zeros. This model operates in two stages: the first stage determines whether the count is zero or positive using a binary classification, while the second stage models only the positive counts, often employing a regression models [1].

Following the hurdle-type modeling, in this Notebook, we will train and evaluate a Random Forest (RF) regression model to predict the Asthma Mortality Rate (AMR) values where it is present  (AMR_BIN == 1 or AMR values > 0). We will apply a walk-forward (expanding window) validation approach, which is appropriate for epidemiological studies involving time series data. We will start by training the RF regression model using data from 2001 to 2006 (a 5-year window) and testing it with data from 2007. The training window will then be expanded by one year at each iteration until it spans from 2001 to 2021, with 2022 data used for testing.

## 📚 Import required libraries

In [ ]:
# dataframe libraries
import pandas as pd
import numpy as np

# geospatial libraries
import geopandas as gpd
import mapclassify
from libpysal.weights import Queen
from esda.moran import Moran
from pysal.explore import esda

# plot libraries
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# sklearn libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score,explained_variance_score,median_absolute_error, max_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.model_selection import ParameterSampler

# other libraries
import re
import os
from joblib import Parallel, delayed
import shap
from itables import init_notebook_mode, show
from lime.lime_tabular import LimeTabularExplainer
import itertools

In [ ]:
%cd work/

In [ ]:
# Set the PROJ_LIB path
os.environ['PROJ_LIB'] = "/opt/conda/envs/gds/share/proj"

In [ ]:
 # Copy-on-Write
 pd.options.mode.copy_on_write = True

##🗂 Funtions to load and transform the dataset

In [ ]:
# path to gdf
path = "pdt/asthma_mortality/data/gpkg/data.gpkg"

In [ ]:
# Funtion to create lag variables (up to 2 years)
def create_lags(df, var, max_lag=2):
  """
  Create lagged variables (up to 2 years).

    Parameters:
    df (pd.DataFrame): Original DataFrame.
    var (str): Name of the variable to create lags for.

    Returns:
    df (pd.DataFrame): DataFrame with lagged variables.
  """

  for lag in range(1, max_lag+1):
    df[f"{var}_lag{lag}"] = df[var].shift(lag)
  return df

In [ ]:
def create_ts_long_format(path):
  """
  Create long-format time series dataset from GeoDataFrame.

    Parameters:
    path (str): Path to the GeoPackage file containing the GeoDataFrame.

    Returns:
    df_ts (pd.DataFrame): Long-format time series dataset.
  """

  # Load dataset with data per department
  gdf = gpd.read_file(path)

  # Reshape df to ts long format
  years = range(2001, 2023)
  records = []
  # iterate rows
  for _, row in gdf.iterrows():
    iddpto = row["IDDPTO"]
    for year in years:
        records.append({
            "IDDPTO": iddpto,
            "YEAR": year,
            "AMR": row.get(f"AMR_{year}", np.nan),
            "PM25": row.get(f"PM25_{year}", np.nan),
            "PM10": row.get(f"PM10_{year}", np.nan),
            "NBA": row.get(f"NBA_{year}", np.nan),
            "PD": row.get(f"PD_{year}", np.nan),
            "NAGRT": row.get(f"NAGRT_{year}", np.nan),
            "NNWVT": row.get(f"NNWVT_{year}", np.nan),
            "NBUT": row.get(f"NBUT_{year}", np.nan)
            })

  # create new df from list
  df_ts = pd.DataFrame(records)

  # Create lagged variables
  for var in ["NAGRT", "NNWVT", "NBUT"]:
    df_ts = create_lags(df_ts, var)

  # Drop the initial rows with NaNs due to lagging
  df_ts = df_ts.dropna().reset_index(drop=True)

  return df_ts

## ⚙️  Training and Evaluating Random Forest Regression Model with Default Hiperparameters (Walk-forward)

In [ ]:
# apply the funtion to create long-format time series dataset from GeoDataFrame
df_ts = create_ts_long_format(path)

In [ ]:
# Binary target
df_ts['AMR_BIN'] = (df_ts['AMR'] > 0).astype(int)

In [ ]:
# Filter AMR_BIN result to keep only rows where AMR_BIN is 1 (occurrence of asthma mortality)
bin_positive = df_ts[(df_ts['AMR_BIN'] == 1)]

In [ ]:
# Display the filtered DataFrame
init_notebook_mode(all_interactive=True)
show(bin_positive)

In [ ]:
# Training and Testing RF Regression Models
results = []  # store results

for i in range(2006, 2022):  # walk-forward from 2006 to 2021
    train_years = list(range(2001, i + 1))  # expanding window
    test_year = i + 1
    print(f"Training: {train_years[0]}–{train_years[-1]}, Testing: {test_year}")

    # Filter by year
    train_df = df_ts[df_ts['YEAR'].isin(train_years)]
    test_df = df_ts[df_ts['YEAR'] == test_year]

    # Filter AMR_BIN == 1
    train_pos = train_df[train_df['AMR_BIN'] == 1].copy()
    test_pos = test_df[test_df['AMR_BIN'] == 1].copy()

    # Skip iteration if empty (avoid errors)
    if train_pos.empty or test_pos.empty:
        print(f"Skipped: No positive cases in train or test for {test_year}")
        continue

    # Features and regression target
    features = ['PM25', 'PM10', 'NBA', 'PD', 'NAGRT', 'NNWVT', 'NBUT',
                'NAGRT_lag1', 'NAGRT_lag2', 'NNWVT_lag1', 'NNWVT_lag2',
                'NBUT_lag1','NBUT_lag2']
    target = 'AMR'  # regression target


    X_train = train_pos[features]
    y_train = train_pos[target]
    X_test = test_pos[features]
    y_test = test_pos[target]

    # Train regression model with default hiperparameters
    model = RandomForestRegressor(n_jobs=-1, random_state=42)
    model.fit(X_train, y_train)

    # Predict and evaluate
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred, squared=False)

    # Save results
    results.append({
        'train_years': f"{train_years[0]}-{train_years[-1]}",
        'test_year': test_year,
        'MAE': mae,
        'RMSE': rmse,
        'R2': r2
    })


In [ ]:
# convert results into a data frame
results_rf_reg = pd.DataFrame(results)

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
results_rf_reg

In [ ]:
# Calculate and display the mean and standard deviation for each evaluation metric
mean_metrics = results_rf_reg[['R2', "MAE", "RMSE"]].mean()
std_metrics = results_rf_reg[['R2', "MAE", "RMSE"]].std()

print("Mean of Evaluation Metrics:")
print(mean_metrics)
print("\nStandard Deviation of Evaluation Metrics:")
print(std_metrics)

The evaluation metrics indicate that the Random Forest regression model shows weak and somewhat unstable predictive performance, with a mean R² of **≈ 0.31 ± 0.19**, indicating that the model explains approximately one-third of the variance in asthma mortality rate across departments. Prediction errors are relatively high (**MAE ≈ 1.57 ± 0.26**, **RMSE ≈ 2.74 ± 0.83**), suggesting limited accuracy in estimating asthma mortality rates. The larger RMSE relative to MAE indicates the presence of some  residual errors, particularly for departments with extreme values. Moreover, the relatively high standard deviation of R² highlights variability in model performance across different training–testing splits. Overall, the model captures some underlying patterns but remains inconsistent, with  limitations in predicting heterogeneous or extreme asthma mortality outcomes.


##🛠️ Parameter Tuning Random Forest Regression Model (Walk-forward)

In this section will perform hyperparameter tunning on the RF regression model. We will use the Random Search method because Standard k-fold CV randomly shuffles observations, which could introduces data leakage in panel/time-series data [2]. We will tune the following hyperparameters because it has been reported as the most frequent RF-tuned hyperparameters [3],[4],[5]:

* n_estimators: The number of trees in the forest. Tuning it identifies the point where predictive stability is achieved efficiently (bias–variance trade-off )
* max_depth: The maximum depth of the tree. Tuning it finds the depth at which the model generalises across unseen years without memorising fold-specific patterns (overfitting).
* min_samples_leaf: The minimum number of samples required to be at a leaf node. Tuning it balances sensitivity to  epidemiological heterogeneity against robustness to small-sample instability.

In [ ]:
# apply the funtion to create long-format time series dataset from GeoDataFrame
df_ts_pt = create_ts_long_format(path)

In [ ]:
# Binary target
df_ts_pt['AMR_BIN'] = (df_ts_pt['AMR'] > 0).astype(int)

In [ ]:
# Predictor variables: air quality, demographic, and lagged epidemiological features
features = [
    'PM25', 'PM10', 'NBA', 'PD', 'NAGRT', 'NNWVT', 'NBUT',
    'NAGRT_lag1', 'NAGRT_lag2',
    'NNWVT_lag1', 'NNWVT_lag2',
    'NBUT_lag1',  'NBUT_lag2'
]

# Regression target
target = 'AMR'

# Discrete hyperparameter grid for Random Search
# All combinations will be evaluated — grid is intentionally small for exhaustive coverage
param_grid = {
    #'n_estimators'     : [100, 200, 300, 500],
    'n_estimators'     : [200, 250, 300, 350],
    'max_depth'        : [10, 20, 30, None],
    'min_samples_leaf' : [1, 3, 5],
    #'max_features'     : [0.1, 0.5, 1.0]
}

# Generate all hyperparameter combinations
param_combinations = [
    dict(zip(param_grid.keys(), values))
    for values in itertools.product(*param_grid.values())
]

print(f"Total hyperparameter combinations per fold: {len(param_combinations)}")

In [ ]:
def evaluate_params(params, X_train, y_train, X_test, y_test):
    """
    Train a Random Forest with a given hyperparameter set and return evaluation metrics.
    Designed to be called in parallel via joblib.Parallel.

    Parameters
    ----------
    params   : dict  — hyperparameter combination to evaluate
    X_train  : array — training feature matrix
    y_train  : array — training target vector
    X_test   : array — test feature matrix
    y_test   : array — test target vector

    Returns
    -------
    dict with params, MAE, RMSE, and R²
    """
    # Instantiate model with current hyperparameter combination
    model = RandomForestRegressor(
        n_estimators      = params['n_estimators'],
        max_depth         = params['max_depth'],
        min_samples_leaf  = params['min_samples_leaf'],
        n_jobs            = -1,       # Parallelise tree fitting within each model
        random_state      = 42
    )

    # Fit on expanding training window
    model.fit(X_train, y_train)

    # Predict on the held-out test year
    y_pred = model.predict(X_test)

    # Compute evaluation metrics
    mae  = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2   = r2_score(y_test, y_pred)

    return {'params': params, 'MAE': mae, 'RMSE': rmse, 'R2': r2}

In [ ]:
# Store best result per expanding window fold
results = []

# Walk-forward expanding window: start training 2001–2006, test 2007
# Expands annually until training 2001–2021, test 2022
for test_year in range(2007, 2023):

    # Define expanding training window (all years up to and including test_year - 1)
    train_years = list(range(2001, test_year))

    print(f"Training: {train_years[0]}–{train_years[-1]}  |  Testing: {test_year}")

    # Filter to positive AMR cases only (AMR_BIN == 1), consistent with prior workflow
    train_pos = df_ts_pt[df_ts_pt['YEAR'].isin(train_years) & (df_ts_pt['AMR_BIN'] == 1)].copy()
    test_pos  = df_ts_pt[(df_ts_pt['YEAR'] == test_year)    & (df_ts_pt['AMR_BIN'] == 1)].copy()

    # Skip fold if either partition is empty to avoid fitting errors
    if train_pos.empty or test_pos.empty:
        print(f"  Skipped: No positive cases in train or test for {test_year}")
        continue

    # Extract feature matrices and target vectors
    X_train = train_pos[features].values
    y_train = train_pos[target].values
    X_test  = test_pos[features].values
    y_test  = test_pos[target].values

    # Evaluate all hyperparameter combinations in parallel
    # n_jobs=-1 distributes combinations across all available CPU cores
    # prefer='threads' avoids joblib overhead for lightweight RF calls;
    # switch to 'processes' if memory allows and RF n_jobs is set to 1 above
    fold_results = Parallel(n_jobs=-1, prefer='threads')(
        delayed(evaluate_params)(params, X_train, y_train, X_test, y_test)
        for params in param_combinations
    )

    # Select the best combination: primary = lowest RMSE, secondary = highest R²
    # Sort ascending by RMSE, then descending by R² to break ties
    fold_results_sorted = sorted(
        fold_results,
        key=lambda x: (round(x['RMSE'], 6), -round(x['R2'], 6))
    )
    best = fold_results_sorted[0]

    # Store result for this fold
    results.append({
        'train_years' : f"{train_years[0]}-{train_years[-1]}",
        'test_year'   : test_year,
        'MAE'         : round(best['MAE'],  6),
        'RMSE'        : round(best['RMSE'], 6),
        'R2'          : round(best['R2'],   6),
        'Best_Params' : best['params']
    })

In [ ]:
# Assemble results into a summary DataFrame
results_rf_reg_ht = pd.DataFrame(results)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
results_rf_reg_ht

In [ ]:
# Calculate and display the mean and standard deviation for each evaluation metric
mean_metrics = results_rf_reg_ht[['R2', "MAE", "RMSE"]].mean()
std_metrics = results_rf_reg_ht[['R2', "MAE", "RMSE"]].std()

print("Mean of Evaluation Metrics:")
print(mean_metrics)
print("\nStandard Deviation of Evaluation Metrics:")
print(std_metrics)

The evaluation metrics indicate that the Random Forest regression model achieves **moderate predictive performance**, with a mean **R² of ≈ 0.43 ± 0.12**, suggesting that it explains approximately 43% of the variance in asthma mortality rates across departments. Prediction errors remain moderately high (**MAE ≈ 1.51 ± 0.23**, **RMSE ≈ 2.50 ± 0.69**), indicating some limitations in accurately estimating mortality rates. However, the smaller gap between MAE and RMSE suggests fewer extreme residual errors compared with the model iterations without hyperparameter tuning. The lower standard deviation across all metrics, particularly for R², indicates more consistent performance across training–testing splits. Overall, with hyperparameter tuning the model captures meaningful patterns in the data and demonstrates improved stability.

## 🏋️‍♂️Training Random Forest Regression Model on 2001–2021 Data and Predicting 2022 Using Tuned Hyperparameters

In this section, for the SHAP analysis and mapping, we will train the RF regression model on data from 2001–2021 and predict the 2022 Normalized Asthma Mortality Rate in departments where it is present (CA_bin = 1 or NAMR > 0). We will use the tuned hyperparameters obtained in the previous step.

In [ ]:
# apply the funtion to create long-format time series dataset from GeoDataFrame
df_ts_0122 = create_ts_long_format(path)

In [ ]:
# Binary target
df_ts_0122['AMR_BIN'] = (df_ts_0122['AMR'] > 0).astype(int)

In [ ]:
# Define features and target
features = ['PM25', "PM10", 'NBA', 'PD', 'NAGRT', 'NNWVT', 'NBUT',
                'NAGRT_lag1', 'NAGRT_lag2', 'NNWVT_lag1', 'NNWVT_lag2',
                'NBUT_lag1','NBUT_lag2']
target = 'AMR'

In [ ]:
# Prepare training and test sets
train_df_0122 = df_ts_0122[df_ts_0122['YEAR'] <= 2021].dropna(subset=features + [target])
test_df_0122 = df_ts_0122[df_ts_0122['YEAR'] == 2022].dropna(subset=features + [target])

In [ ]:
# Filter for positive cases (AMR_BIN == 1)
train_pos_0122 = train_df_0122[train_df_0122['AMR_BIN'] == 1].copy()
test_pos_0122 = test_df_0122[test_df_0122['AMR_BIN'] == 1].copy()

In [ ]:
# Define inputs
X_train_0122 = train_pos_0122[features]
y_train_0122 = train_pos_0122[target]

In [ ]:
# Keep IDDPTO and make prediction-ready copy
X_test_full_0122 = test_pos_0122[['IDDPTO'] + features + [target]].copy()
X_test_0122 = X_test_full_0122[features]
y_test_0122 = test_pos_0122[target].reset_index(drop=True)

In [ ]:
#get the tuned hiperpameter of the training 2001-2021
last_row_params = results_rf_reg_ht.iloc[-1]['Best_Params']
formatted_params_0122 = str(last_row_params).replace("'", "").replace(":", "=")
print(formatted_params_0122)

In [ ]:
# Train regression model
reg_0122 = RandomForestRegressor(max_depth= 20,  min_samples_leaf= 3,
                            n_estimators= 200,
                            n_jobs=-1,
                            random_state=42)
reg_0122.fit(X_train_0122, y_train_0122)

In [ ]:
# Predict using feature columns
y_pred_0122 = reg_0122.predict(X_test_0122)

In [ ]:
# Store prediction in column AMR_PRED (Predicted Asthma Mortality rate)
X_test_full_0122['AMR_PRED'] = y_pred_0122

In [ ]:
# Evaluate
r2 = r2_score(y_test_0122, y_pred_0122)
mae = mean_absolute_error(y_test_0122, y_pred_0122)
rmse = mean_squared_error(y_test_0122, y_pred_0122, squared=False)

In [ ]:
# Print evaluation
print("Prediction Results for 2022:")
print(f"R²: {r2:.3f}, MAE: {mae:.3f}, RMSE: {rmse:.3f}")

Metric Value	Interpretation:

The tuned Random Forest regression model, trained on departments with positive asthma mortality cases (AMR_BIN = 1), achieved an R² of 0.589. This value indicates that the model accounted for 58.9% of the variability in asthma mortality rates among departments where mortality occurred. The result suggests moderate predictive performance for estimating 2022 asthma mortality rates in this subset.

In [ ]:
# select IDDPTO, AMR and AMR_PRED from X_test_full
reg_pred_result_0122 = X_test_full_0122[['IDDPTO', 'AMR', 'AMR_PRED']].copy()

In [ ]:
# change column names: "AMR" to "AMR_2022", "AMR_PRED" to "AMR_PRED_2022"
reg_pred_result_0122.rename(columns={'AMR': 'AMR_2022', 'AMR_PRED': 'AMR_PRED_2022'}, inplace=True)

In [ ]:
# visualize the results
init_notebook_mode(all_interactive=True)
reg_pred_result_0122

In [ ]:
# create a scatterplot between AMR_2022 vs AMR_PRED_2022
plt.figure(figsize=(10, 6))
plt.scatter(reg_pred_result_0122['AMR_2022'], reg_pred_result_0122['AMR_PRED_2022'])
plt.xlabel('AMR_2022 (Actual)')
plt.ylabel('AMR_2022_PRED (Predicted)')
plt.title('Scatter Plot of Actual vs. Predicted Asthma Mortality Rate (2022)')
plt.grid(True)
plt.show()

The scatter plot indicates a moderate positive relationship between observed and predicted asthma mortality rates, suggesting that the model captures the general direction of variation in asthma mortality rate.

## 🤔 SHAP (SHapley Additive Explanations)

In this section we will interpret the contribution of each independent variable to the final predition of the RF regression model using the SHAP method

In [ ]:
# SHAP for Regression
# Use TreeExplainer (optimized for RandomForest)
explainer_0122 = shap.TreeExplainer(reg_0122)

# get test set
X_0122 = test_pos_0122[features]

# Function to compute SHAP values for a single row
def compute_shap(row):
    # For regression, shap_values returns a single 1D array
    return explainer_0122.shap_values(row)

# Parallel computation (adjust n_jobs as needed)
shap_values_list_0122 = Parallel(n_jobs=-1)(
    delayed(compute_shap)(X_0122.iloc[[i]]) for i in range(len(X_0122))
)

# Stack the 1D SHAP value arrays vertically
# No need to index [:, 1] as there's only one set of SHAP values for regression
shap_values_0122 = np.vstack(shap_values_list_0122)

In [ ]:
plt.tight_layout?

In [ ]:
# Generate a SHAP summary plot to visualize the impact of features on the model's predictions
shap.summary_plot(shap_values_0122, X_0122, feature_names=features,
                  show= False,  cmap = "plasma")
plt.title("SHAP Summary Plot RF Regression", fontsize=17)
plt.xlabel("SHAP Values  (rate)", size=12)
plt.ylabel("Features", size=12)

# adjust the padding between and around subplots.
plt.tight_layout()

# save the chart with 300 dpi
plt.savefig('pdt/asthma_mortality/data/images/figures/shap_rf_reg.png', dpi=300)

plt.show();

### Extract SHAP Values

In [ ]:
# SHAP values as dataframe
shap_df_0122 = pd.DataFrame(
    shap_values_0122,
    columns=features,
    index=X_0122.index
)

In [ ]:
# rename all columns of shap_df like _sahp
shap_df_0122.columns = [f"{col}_SHAP" for col in shap_df_0122.columns]

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
shap_df_0122.head()

In [ ]:
# X datagrame
X_df_0122 = X_0122.copy()

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
X_df_0122.head()

In [ ]:
# join shap_df_0122 and X_df_0122 by index
shap_df_join_0122 = shap_df_0122.join(X_df_0122)

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
shap_df_join_0122.head()

#### Population Density (PD)

In [ ]:
# from shap_df_join_0122 select all records where PD_SHAP is greater than 0
pd_purple_pos_0122 =  shap_df_join_0122[(shap_df_join_0122['PD_SHAP'] > 0)][["PD_SHAP", "PD"]]

In [ ]:
# get statistics
init_notebook_mode(all_interactive=True)
pd_purple_pos_0122.describe()

In [ ]:
# from shap_df_join select all records where PD_SHAP is between 0 and -3
pd_purple_neg_low_0122 =  shap_df_join_0122[(shap_df_join_0122['PD_SHAP'] >= -3.0) &  (shap_df_join_0122['PD_SHAP'] <= 0)][["PD_SHAP", "PD"]]

In [ ]:
# get statistics
init_notebook_mode(all_interactive=True)
pd_purple_neg_low_0122.describe()

In [ ]:
# from shap_df_join select all records where PD_SHAP is between -2.5 and -3.0
pd_purple_neg_high_0122 =  shap_df_join_0122[(shap_df_join_0122['PD_SHAP'] >= -4.0) &  (shap_df_join_0122['PD_SHAP'] < -3.0)][["PD_SHAP", "PD"]]

In [ ]:
# get statistics
init_notebook_mode(all_interactive=True)
pd_purple_neg_high_0122.describe()

#### Particulate Matter ($PM_{10}$)

In [ ]:
# from shap_df_join_0122 select all records where PM10_SHAP is greater than 0
pm10_pos_0122 =  shap_df_join_0122[(shap_df_join_0122['PM10_SHAP'] > 0)][["PM10_SHAP", "PM10"]]

In [ ]:
# get statistics
init_notebook_mode(all_interactive=True)
pm10_pos_0122.describe()

In [ ]:
# from shap_df_join_0122 select all records where PM10_SHAP is less than 0
pm10_neg_0122 =  shap_df_join_0122[(shap_df_join_0122['PM10_SHAP'] <= 0)][["PM10_SHAP", "PM10"]]

In [ ]:
# get statistics
init_notebook_mode(all_interactive=True)
pm10_neg_0122.describe()

In [ ]:
# Get baseline (expected value-mean model prediction)
baseline_0122 = explainer_0122.expected_value

print("Model baseline (expected value):", baseline_0122)



The SHAP summary plot indicates that population density (PD) is the dominant predictor of asthma mortality rates in the Random Forest regression model. PD has the widest distribution of SHAP values, suggesting a strong, heterogeneous, and non-linear contribution to deviations from the model’s expected value (≈ 3.43). Environmental and land-use variables, including $PM_{10}$, have smaller contributions that appear more linear.

**Population Desity**

* First, observations with low PD values (mean PD ≈ 6.51; purple points) and positive SHAP values (mean SHAP ≈ 2.34) increase the predicted asthma mortality rate. For these locations, the contribution of PD raises the prediction from the baseline of 3.43 to approximately 5.77. This pattern suggests that very sparsely populated areas with observed mortality are associated with higher model predictions. It may reflect limited access to healthcare services and longer emergency response times in sparsely populated areas [6]. Rural areas may also have limited access to healthcare facilities and professionals, potentially delaying diagnosis and treatment and contributing to higher mortality rates [7]. However, the pattern may also reflect the “small counts” issue common in epidemiological studies, in which low population numbers can produce unstable or unreliable rate estimates [8].

* A second subset comprises observations with high PD values (mean PD ≈ 1126.25; purple points) and negative SHAP values ranging from 0 to −3.0 (mean SHAP ≈ −1.64). In this range, PD reduces the predicted AMR to approximately 1.79. This pattern suggests that higher population density is associated with lower AMR predictions in the model.

* Third, a subset of observations has very high PD values (mean PD ≈ 7359.78; yellow points) and negative SHAP values between −4.0 and −3.0 (mean SHAP ≈ −3.15). For this subset, PD reduces the predicted AMR to approximately 0.28. These results suggest that highly urbanized or densely populated areas are associated with lower predicted AMR in the model.

**Particulate Matter ($PM_{10}$)**


* $PM_{10}$ shows both positive and negative contributions to predicted asthma mortality rates. High $PM_{10}$ concentrations (mean ≈ 126.24) are associated with positive SHAP values (mean ≈ 0.26), increasing the prediction from the baseline of 3.43 to approximately 3.69. In contrast, $PM_{10}$ concentrations with a mean of approximately 115.74 are associated with negative SHAP values (mean ≈ −0.25), decreasing the prediction from the baseline of 3.43 to approximately 3.18. These results suggest that the contribution of $PM_{10}$ to AMR predictions is moderate and varies across observations in the context of the other model variables.

Overall, the SHAP analysis identifies PD as the dominant non-linear predictor in the Random Forest regression model, with contributions that vary across population-density regimes. Low population density is associated with higher predicted AMR, whereas higher density is associated with lower predicted AMR; the most negative SHAP values occur in highly populated areas. This pattern suggests that PD may capture contextual factors that influence model predictions and supports further sensitivity analyses to distinguish demographic effects from underlying environmental and exposure-related drivers.




## 🗣️ LIME (Local Interpretable Model-agnostic Explanations)

LIME (Local Interpretable Model-agnostic Explanations) explains individual predictions from "black-box" models such as Random Forests by approximating their behavior locally with simpler models (e.g., linear models) [9].

In [ ]:
# Initialize LIME Tabular Explainer (Regression mode)
lime_explainer_reg = LimeTabularExplainer(
    training_data=X_train_0122.values,
    feature_names=X_train_0122.columns.tolist(),
    mode="regression",
    discretize_continuous=True,
    random_state=42
)

In [ ]:
# LIME explanations
lime_results = []

for idx in range(len(X_test_0122)):
    exp = lime_explainer_reg.explain_instance(
        data_row = X_test_0122.iloc[idx].values,
        predict_fn = reg_0122.predict,
        num_features=len(X_test_0122.columns)
    )

    for feature, contribution in exp.as_list():
        lime_results.append({
            "IDDPTO": test_pos_0122.iloc[idx]["IDDPTO"],
            "feature": feature,
            "lime_contribution": contribution
        })

In [ ]:
# convert results into a dataframe
lime_reg_contributions_df = pd.DataFrame(lime_results)

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
lime_reg_contributions_df

In [ ]:
# Aggregate LIME contributions by department and predictor
lime_reg_by_dept = (
    lime_reg_contributions_df
    .groupby(['IDDPTO', 'feature'], as_index=False)
    .agg(
        mean_lime_contribution=('lime_contribution', 'mean'),# Mean contribution per predictor per department
        abs_mean_contribution=('lime_contribution', lambda x: np.mean(np.abs(x)))
    )
)

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
lime_reg_by_dept

In [ ]:
# Pivot to wide format (mapping )
lime_reg_pivot = lime_reg_by_dept.pivot(
    index='IDDPTO',
    columns='feature',
    values='mean_lime_contribution'
).reset_index()

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
lime_reg_pivot

### Mapping LIME contributions

In this section, we map the LIME contributions of Population Density (PD), and $PM_{10}$. PD and $PM_{10}$ were identified by SHAP as the most influential predictors driving the Random Forest predictions,. Mapping these local contributions allows us to explore the spatial heterogeneity of predictor effects and identify the regions where each variable acts as a positive or negative driver of asthma mortality predictions.

In [ ]:
# Load dataset with data per department
gdf = gpd.read_file("pdt/asthma_mortality/data/gpkg/data.gpkg")

In [ ]:
# Merge LIME pivot with geodataframe
gdf_lime_reg = gdf.merge(
    lime_reg_pivot,
    on='IDDPTO',
    how='left'
)

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
gdf_lime_reg

In [ ]:
# gdf_lime_reg length
len(gdf_lime_reg)

### Population Density (PD)

In [ ]:
def extract_range_for_variable(s, variable_name):
    """
    Extracts the lower and upper bounds from a range string for a given variable.
    Returns a tuple (priority, lower_bound, upper_bound) for custom sorting.
    """
    numbers = [float(n) for n in re.findall(r'\d+\.?\d*', s)]

    # Check for 'X < VAR <= Y' first (most specific range pattern)
    if f'< {variable_name} <=' in s:
        # Priority 1 for 'X < variable <= Y' ranges, then (lower_bound=number1, upper_bound=number2)
        # Expected two numbers (lower and upper bounds)
        if len(numbers) >= 2:
            return (1, numbers[0], numbers[1])
        else:
            # Handle malformed string: treat as undefined range for sorting
            return (1, float('-inf'), float('inf'))
    # Then check for 'VAR <= X'
    elif f'{variable_name} <=' in s:
        # Priority 0 for '<=' ranges, then (lower_bound=-inf, upper_bound=number)
        # Expected one number (upper bound)
        if numbers:
            return (0, float('-inf'), numbers[0])
        else:
            # Handle malformed string: treat as very low upper bound for sorting
            return (0, float('-inf'), float('-inf'))
    # Then check for 'VAR > X'
    elif f'{variable_name} >' in s:
        # Priority 2 for '>' ranges, then (lower_bound=number, upper_bound=+inf)
        # Expected one number (lower bound)
        if numbers:
            return (2, numbers[0], float('inf'))
        else:
            # Handle malformed string: treat as very high lower bound for sorting
            return (2, float('inf'), float('inf'))
    else:
        # Fallback for truly unexpected formats with lowest priority
        return (3, float('-inf'), float('inf'))

In [ ]:
# Extract LIME ranges for 'PD' feature
pd_long_columns = [col for col in gdf_lime_reg.columns if 'PD' in col and len(col) > 7]

# Sort the list programmatically using the custom key for 'PD'
pd_long_columns_sorted = sorted(pd_long_columns, key=lambda s: extract_range_for_variable(s, 'PD'))

print(pd_long_columns_sorted)

In [ ]:
# copy pd_long_columns
pd_long_columns_titles = pd_long_columns_sorted.copy()

# add "intermediate-density departments:" to pd_long_columns[0], and  "high-density departments:" to pd_long_columns[1]
pd_long_columns_titles[0] = "Low\n" + pd_long_columns_sorted[0] + " people/km²"
pd_long_columns_titles[1] = "Lower-Intermediate\n" + pd_long_columns_sorted[1] + " people/km²"
pd_long_columns_titles[2] = "Upper-Intermediate\n" + pd_long_columns_sorted[2] + " people/km²"
pd_long_columns_titles[3] = "High\n" + pd_long_columns_sorted[3] + " people/km²"

In [ ]:
def plot_lime_contributions(columns, titles, gdf_lime_reg, overall_title):
    """
    Generates and displays a set of choropleth maps showing LIME contributions
    for different ranges of a given variable.

    Args:
        columns (list): A list of column names in gdf_lime_reg to be mapped.
        titles (list): A list of titles for each subplot, corresponding to `columns`.
        gdf_lime_reg (gpd.GeoDataFrame): The GeoDataFrame containing LIME contributions
                                        and geometry.
        overall_title (str): The main title for the entire figure.
    """
    # Calculate global min and max across all relevant columns for the common color bar
    global_min = gdf_lime_reg[columns].min().min()
    global_max = gdf_lime_reg[columns].max().max()

    # Create a figure and a grid of subplots
    fig, axes = plt.subplots(1, len(columns), figsize=(len(columns) * 4.0, 10))
    axes = axes.flatten()

    # Create a ScalarMappable for the common colorbar
    current_cmap = mpl.colormaps.get_cmap("viridis")
    sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=global_min, vmax=global_max), cmap=current_cmap)
    sm._A = []

    for ax, col, title in zip(axes, columns, titles):
        gdf_valid = gdf_lime_reg[gdf_lime_reg[col].notna()]
        gdf_nan = gdf_lime_reg[gdf_lime_reg[col].isna()]

        # Plot departments with NaN values (outline only)
        gdf_nan.plot(
            ax=ax,
            facecolor="none",
            edgecolor="black",
            linewidth=0.5
        )

        # Plot departments with valid LIME values
        gdf_valid.plot(
            ax=ax,
            column=col,
            cmap="viridis",
            linewidth=0.5,
            edgecolor="black",
            legend=False,
            vmin=global_min,
            vmax=global_max
        )

        ax.set_title(title, fontsize=10)
        ax.set_axis_off()

    # Add a single colorbar for the entire figure on the right
    cbar_ax = fig.add_axes([0.92, 0.15, 0.01, 0.7])
    cbar = fig.colorbar(
        sm,
        cax=cbar_ax,
        orientation="vertical"
    )
    cbar.ax.tick_params(labelsize=8)
    cbar.locator = MaxNLocator(nbins=4)
    cbar.update_ticks()
    cbar.set_label("LIME contributions", fontsize=8, rotation=270, labelpad=15)

    # Add the overall title
    plt.suptitle(overall_title, fontsize=14, y=0.85)

    # Create 'No Data' patch for the legend
    no_data_patch = Patch(facecolor='none', edgecolor='black', label='No Data')

    # Add the 'No Data' legend in the lower center of the figure
    fig.legend(handles=[no_data_patch], loc='lower center', bbox_to_anchor=(0.5, 0.2), fontsize=10, frameon=False)

    # save the chart with 300 dpi
    #plt.savefig('pdt/asthma_mortality/data/images/figures/lime_pm10_rf_reg.png', dpi=300)

    plt.show()

In [ ]:
# Map LIME contributions for 'PD'
columns_to_map_pd = pd_long_columns_sorted

# define subplot titles
titles_for_subplots_pd = pd_long_columns_titles

# Define the overall title
overall_plot_title_pd = 'LIME contributions of PD for Predicted Asthma Mortality Rate in 2022 Argentina'

# Call the function to generate the plot
plot_lime_contributions(columns_to_map_pd, titles_for_subplots_pd, gdf_lime_reg, overall_plot_title_pd)

The maps indicate a clear separation between **PD level** (interval membership) and **PD local effect** (LIME sign/magnitude). The lowest interval (**PD ≤ 8.46**) is dominated by yellow tones, showing mostly **positive** contributions, often large (upper end of the color scale). These departments are spatially dispersed but forming visible pockets in northern and central parts of the country, with additional isolated southern units. In this interval, PD acts as a recurrent positive driver of predicted AMR.

For **8.46 < PD ≤ 23.10**, contributions shift toward **near-zero to mildly negative** (teal/blue). Spatially, these units appear more concentrated in a north-central/east-central belt, with fewer strong effects. This looks like a transition band: PD still contributes locally, but with weaker and less uniformly positive influence than in the lowest interval.

For **23.10 < PD ≤ 184.29**, dark blue/purple tones dominate, indicating **negative** contributions across many municipalities. Negative effects cluster in several contiguous areas (notably north-central and eastern sectors), with intensity varying from moderate to strong. In the highest interval (**PD > 184.29**), departm,ents are few and spatially isolated, but contributions are predominantly **strongly negative**, suggesting that very high PD is a consistent local suppressor of RF-predicted AMR in those locations.

The SHAP summary supports this interpretation at the global level: PD is the feature with the widest spread of effects, including extreme positive outliers and substantial negative values. Combined with the maps, this suggests an overall gradient from positive effects at very low PD to negative effects at moderate/high PD, while still allowing regional heterogeneity in magnitude within the same PD class.

### $PM_{10}$

In [ ]:
# Extract LIME ranges for 'PD' feature
pm10_long_columns = [col for col in gdf_lime_reg.columns if 'PM10' in col and len(col) > 9]

# Sort the list programmatically using the custom key for 'PM10'
pm10_long_columns_sorted = sorted(pm10_long_columns, key=lambda s: extract_range_for_variable(s, 'PM10'))

print(pm10_long_columns_sorted)

In [ ]:
# copy pm10_long_columns
pm10_long_columns_titles = pm10_long_columns_sorted.copy()

# add category names
pm10_long_columns_titles[0] = "Low\n" + pm10_long_columns_titles[0] + " μg/m³"
pm10_long_columns_titles[1] = "Lower-Intermediate\n" + pm10_long_columns_titles[1] + " μg/m³"
pm10_long_columns_titles[2] = "Upper-Intermediate\n" + pm10_long_columns_titles[2] + " μg/m³"
pm10_long_columns_titles[3] = "High\n" + pm10_long_columns_titles[3] + " μg/m³"

In [ ]:
# Map LIME contributions for 'PM10'
columns_to_map_pm10 = pm10_long_columns_sorted

# define subplot titles
titles_for_subplots_pm10 = pm10_long_columns_titles

# Define the overall title
overall_plot_title_pm10 = 'LIME contributions of $PM_{10}$ for for Predicted Asthma Mortality Rate in 2022 Argentina'

# Call the function to generate the plot
plot_lime_contributions(columns_to_map_pm10, titles_for_subplots_pm10, gdf_lime_reg, overall_plot_title_pm10)

The maps show a directional gradient in local $PM_{10}$ effects across interval classes. In the lowest class ($PM_{10}$ ≤ 116.03 μg/m³), most colored municipalities are purple to dark blue, indicating negative LIME contributions. These negatives are spatially widespread, with visible clusters in northern and western sectors and some southern outliers. This pattern indicates that low $PM_{10}$ values often reduce the local RF prediction of AMR in the departments where the mortality event occurred:

In the next class (116.03 < $PM_{10}$ ≤ 119.97 μg/m³), contributions move toward near-zero and weakly mixed effects (blue to teal, with occasional light green). Spatially, these municipalities are more concentrated in central and eastern belts, and the contribution magnitudes are smaller than in the first class. This interval behaves as a transition zone, where $PM_{10}$ has limited local influence in many areas and a modest negative tendency in others.

For higher concentrations, the sign reverses. In 119.97 < $PM_{10}$ ≤ 127.29 μg/m³, green tones dominate, showing positive local contributions in a cluster centered in east-central Argentina. In the highest class ($PM_{10}$ > 127.29 μg/m³), departments are fewer and geographically concentrated, but colors are yellow to yellow-green, indicating the strongest positive contributions. The SHAP summary is consistent with this: $PM_{10}$ has a narrower global effect range than PD, but higher $PM_{10}$ values align with positive contributions and lower $PM_{10}$ values with negative contributions. Similar $PM_{10}$ intervals are mostly consistent in sign, while magnitude still varies by location.
```

## 🌍 Mapping actual vs predicted asthma mortality rate

As explained above, since our dataset contains a high proportion of zero values, we will map the predicted AMR for 2022 only in departments where the actual values are greater than zero. Departments with an actual AMR value of zero will retain a predicted value of zero.

### Preparing data for mapping

In [ ]:
# apply the funtion to create long-format time series dataset from GeoDataFrame
df_ts_map = create_ts_long_format(path)

In [ ]:
# Binary target
df_ts_map['AMR_BIN'] = (df_ts_map['AMR'] > 0).astype(int)

In [ ]:
# from df_ts_map filter 2022 and AMR_BIN equal to 0
dpto_zero = df_ts_map[df_ts_map['YEAR'] == 2022]
dpto_zero = dpto_zero[dpto_zero['AMR_BIN'] == 0]

In [ ]:
# keep IDDPTO and AMR_BIN columns
dpto_zero = dpto_zero[['IDDPTO', 'AMR_BIN']]

In [ ]:
# rename AMR_BIN to AMR and copy the column as AMR_PRED
dpto_zero.rename(columns={'AMR_BIN': 'AMR_2022'}, inplace=True)
dpto_zero['AMR_PRED_2022'] = dpto_zero['AMR_2022'].copy()

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
show(dpto_zero)

In [ ]:
# get df info
dpto_zero.info()

In [ ]:
# concat reg_pred_result_0122 with dpto_zero
df_concat = pd.concat([reg_pred_result_0122, dpto_zero])

In [ ]:
# get df info
df_concat.info()

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
show(df_concat)

In [ ]:
# Load dataset with data per department
gdf = gpd.read_file("pdt/asthma_mortality/data/gpkg/data.gpkg")

In [ ]:
# keep IDDPTO and geometry of the gdf
dpto_geom = gdf[['IDDPTO', 'geometry']]

In [ ]:
# merge df_concat and dpto_geom by IDDPTO preserve  df_concat data
df_map = pd.merge(df_concat, dpto_geom, on='IDDPTO', how='left')

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
show(df_map)

In [ ]:
# round AMR_PRED_2022 to two decimal places
df_map['AMR_PRED_2022'] = df_map['AMR_PRED_2022'].round(2)

In [ ]:
# convert df_map to a gdf
gdf_map = gpd.GeoDataFrame(df_map, geometry='geometry')

In [ ]:
# visualize geodataframe
init_notebook_mode(all_interactive=True)
show(gdf_map)

In [ ]:
# save df_map as gpkg file
#gdf_map.to_file("pdt/asthma_mortality/data/gpkg/results_RF.gpkg", driver="GPKG")

### Calculate classification schema for mapping

We will use [PySAL ](https://pysal.org/)[`mapclassify`](https://pysal.org/mapclassify/index.html) library to determine the most suitable classifier for the choropleth map. The classifier with the lowest ACDM (Mean Absolute Deviation Around the Class Median) will be selected. In PySAL, ACDM measures the goodness of fit of a classification scheme by calculating the average absolute deviation between each data point and the median of its assigned class. Lower ACDM values indicate a better fit between the classifier and the data distribution.

In [ ]:
# open results_RF.gpkg as a gdf
df_cl = gpd.read_file("pdt/asthma_mortality/data/gpkg/results_RF.gpkg")

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
show(df_cl)

In [ ]:
# get df basic info
df_cl.info()

In [ ]:
# Get the length of the dataframe 'df_cl'
len(df_cl)

In [ ]:
# Select data to analize
selected_data = df_cl.loc[:,["AMR_2022", "AMR_PRED_2022"]]

In [ ]:
# exclude AMR_2022 == 0 to calculate map classification schema
selected_data_mc = selected_data[selected_data["AMR_2022"] > 0]

In [ ]:
# Classify the data into 4 quantile groups
q4 = mapclassify.Quantiles(selected_data_mc, k=4)
q4

In [ ]:
# Equal Interval Classification
ei5 = mapclassify.EqualInterval(selected_data_mc, k=5)
ei5

In [ ]:
# Classify the data into groups based on the head/tail breaks algorithm
ht = mapclassify.HeadTailBreaks(selected_data_mc)
ht

In [ ]:
# MaximumBreaks classification method
mb5 = mapclassify.MaximumBreaks(selected_data_mc, k=5)
mb5

In [ ]:
# Apply the Standard Deviation and Mean classification method to the selected data.
msd = mapclassify.StdMean(selected_data_mc)
msd

In [ ]:
# Apply Fisher-Jenks classification with 5 classes
fj5 = mapclassify.FisherJenks(selected_data_mc, k=5)
fj5

ACDM(mean Absolute Deviation Around the class Median) visualization

In [ ]:
# Bunch classifier objects
class5 = q4, ei5, ht, mb5, msd, fj5
# Collect ADCM for each classifier
fits = np.array([c.adcm for c in class5])
# Convert ADCM scores to a DataFrame
adcms = pd.DataFrame(fits)
# Add classifier names
adcms["classifier"] = [c.name for c in class5]
# Add column names to the ADCM
adcms.columns = ["ADCM", "Classifier"]
ax = sns.barplot(
    y="Classifier", x="ADCM", data=adcms, hue= adcms["Classifier"],  legend=False
)
plt.show();

### Create choropleth maps

Two classifiers have the lowest ACDM: FisherJenks and HeadTailBreaks. We'll select FisherJenks as the classifier to create the choropleth maps.

In [ ]:
# Convert the bins to a list for further processing
bins =np.round(fj5.bins, 2).tolist()
bins

In [ ]:
def mapamr_act_pred(gdf_to_plot, colname, title, ax):
    """
    Function to plot actual vs predicted AMR as a choropleth maps

    Args:
        gdf_to_plot (gpd.GeoDataFrame): The GeoDataFrame to plot.
        colname (str): The name of the column to visualize.
        title (str): The title for the subplot.
        ax (matplotlib.axes.Axes): The axes object to plot on.

    Returns:
        tuple: A tuple containing lists of legend handles and labels.
    """
    # Split data:
    # - zero mortality
    # - positive mortality
    gdf_zero = gdf_to_plot[gdf_to_plot[colname] == 0]
    gdf_positive = gdf_to_plot[gdf_to_plot[colname] > 0]

    # Plot zero mortality as hollow polygons
    gdf_zero.plot(
        ax=ax,
        facecolor="none",
        edgecolor="black",
        linewidth=0.5
    )

    # Classification only for positive values
    # `bins` is assumed to be a globally accessible variable from previous cells.
    classification = mapclassify.UserDefined(
        gdf_positive[colname],
        bins=bins
    )

    # Plot positive values without a legend (it will be created centrally)
    classification.plot(
        gdf_positive,
        legend=False,  # Disable legend generation here
        axis_on=False,
        border_color='black',
        border_width=0.5,
        cmap="viridis_r",
        ax=ax
    )

    # Prepare legend handles and labels to be returned
    all_handles = []
    all_labels = []

    # Add custom legend item for zero mortality
    # Changed marker from 'o' (circle) to 's' (square) for a hollow rectangle
    hollow_patch = Line2D(
        [0], [0],
        marker='s', # Changed from 'o' to 's'
        color='black',
        markerfacecolor='white',
        markersize=10,
        linestyle='None',
        label='No Asthma Mortality'
    )
    all_handles.append(hollow_patch)
    all_labels.append("No Asthma Mortality")

    # Get colors for classification bins
    n_bins = len(bins)
    cmap = mpl.colormaps.get_cmap("viridis_r").resampled(n_bins)
    colors = [cmap(i) for i in range(n_bins)]

    # Rounded labels for the bins
    # see non-rounded labels on Fisher-Jenks  results
    bin_labels = [
        "0.2–2",
        "2–3",
        "3–6",
        "6–9",
        "9–14",
    ]

    # Create patches for the classified data and add to handles/labels
    for color, label in zip(colors, bin_labels):
        all_handles.append(Patch(facecolor=color, edgecolor='black', label=label))
        all_labels.append(label)

    # Final formatting for the subplot
    ax.set_title(title)
    ax.set_axis_off()

    return all_handles, all_labels

In [ ]:
# Create a 1x2 (1 row, 2 columns) grid of subplots with a figure size=(12, 8), squeeze=False)
fig, axes = plt.subplots(1, 2, figsize=(12, 8), squeeze=False)

# Add a title to the entire figure with specific font size and position
#plt.suptitle('Observed/Predicted Asthma Mortality Rate (AMR) per 100000 inhabitants\nArgentina 2022', fontsize=14, y=1.0)

# Call maptma for the first plot and collect handles/labels
handles1, labels1 = mapamr_act_pred(gdf_map, "AMR_2022", "Observed AMR", axes[0, 0])

# Call maptma for the second plot and collect handles/labels
handles2, labels2 = mapamr_act_pred(gdf_map, "AMR_PRED_2022", "Predicted AMR", axes[0, 1])

# Combine handles and labels (they should be identical in content but we collect for robustness)
# Since the labels and handles are designed to be the same, we can just use one set.
combined_handles = handles1
combined_labels = labels1

# Create a single figure-level legend at the lower center with 3 columns
fig.legend(
    handles=combined_handles,
    labels=combined_labels,
    loc='lower center',
    bbox_to_anchor=(0.5, 0.05), # Adjust Y to move it below the plots
    ncol=3, # 3 columns
    fontsize=9,
    frameon=False,
    title="AMR in death/100,000 population"
)

# save the chart with 300 dpi
#plt.savefig('pdt/asthma_mortality/data/images/figures/rf_reg_actual_predicted_map.png', dpi=300)

plt.show()

1. **Overall spatial patterns**

   A visual comparison of the observed and predicted AMR maps for Argentina in 2022 indicates substantial spatial variability in both distributions. Areas with higher observed mortality generally correspond to areas with higher predicted AMR, suggesting that the model reproduces broad geographic patterns in the observed data.

2. **Model performance**

   The visual correspondence between the maps suggests that the model captures regional variation, including areas with relatively high and low AMR. However, this comparison alone does not establish the strength of the correlation or validate forecasting performance; these conclusions require quantitative evaluation metrics.

3. **Differences between observed and predicted patterns**

   Although the overall spatial patterns are similar, local discrepancies remain. The predicted map appears to smooth some localized extremes in the observed AMR values, which may indicate that local influences are not fully represented by the modeled variables.

## 📚 References

[1] Hasanah, S., S. Abdullah, and Dian Lestari. "Bayesian method for hurdle regression." ICSA-Int. Conf. Stat. Anal. Vol. 2019. 2019.

[2] S. Albelali and M. Ahmed, “Hidden Leaks in Time Series Forecasting: How Data Leakage Affects LSTM Evaluation Across Configurations and Validation Strategies,” Dec. 07, 2025, arXiv: arXiv:2512.06932. doi: 10.48550/arXiv.2512.06932.


[3] “Effects of tuning decision trees in random forest regression on predicting porosity of a hydrocarbon reservoir. A case study: volve oil field, north sea,” Energy Advances, vol. 3, no. 9, pp. 2335–2347, Sep. 2024, doi: 10.1039/d4ya00313f.


[4] B. F. F. Huang and P. C. Boutros, “The parameter sensitivity of random forests,” BMC Bioinformatics, vol. 17, no. 1, p. 331, Sep. 2016, doi: 10.1186/s12859-016-1228-x.

[5] P. Probst, M. Wright, and A.-L. Boulesteix, “Hyperparameters and Tuning Strategies for Random Forest,” WIREs Data Min & Knowl, vol. 9, no. 3, p. e1301, May 2019, doi: 10.1002/widm.1301.



[6] J. E. Garrett, “Health service accessibility and deaths
from asthma,” Thorax, vol. 52, no. 3, pp. 205–206, 1997,
doi: 10.1136/thx.52.3.205

[7] Thompson, J. A., Mudaranthakam, D. P., &#38; Chollet-Hinton, L. (2023). The rural mortality penalty in U.S. hospital patients with COVID-19. <i>Research Square</i>. https://doi.org/10.21203/rs.3.rs-3467683/v1>

[8] W. S. D. of Health, “Guidelines for using confidence
intervals for public health assessment,” 2012. [Online].
Available: https://doh.wa.gov/sites/default/files/legacy/Documents/1
500/ConfIntGuide.pdf

[9]. Molnar, C. (2020). Interpretable machine learning. Lulu. com







